# Phase 2: YOLOv8n Training Notebook — Diagram-to-Infra-Code

This notebook trains a custom **YOLOv8n** object detection model on synthetic architecture diagram data.

### Target Classes (7 total):
1. `compute` (EC2, EKS)
2. `database` (RDS)
3. `storage` (S3)
4. `load_balancer` (ALB)
5. `network` (VPC/Subnet boundary)
6. `arrow` (directional connection)
7. `text_label` (label box text)

Run this notebook on Google Colab with GPU acceleration enabled (**Runtime -> Change runtime type -> T4 GPU**).

## Setup & Resume Instructions

**If you disconnect during training:**
1. Re-run Steps 1, 2, and 3 to setup the environment and dataset.
2. **Skip Step 4 (Original Training Cell)**.
3. **Run Step 4b (Resume Cell)** to continue training from your Google Drive checkpoint.

## Step 1: GPU Check, Environment Setup & Mount Drive
We mount Google Drive so checkpoints survive runtime disconnects.

In [1]:
!nvidia-smi
!pip install -q ultralytics huggingface_hub opencv-python-headless albumentations pillow

from google.colab import drive
drive.mount('/content/drive')

Mon Sep  7 06:29:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Clone Repository

In [2]:
!git clone https://github.com/Parths-29/Diagram-to-Infra-code.git
%cd Diagram-to-Infra-code

Cloning into 'Diagram-to-Infra-code'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 114 (delta 32), reused 92 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 93.40 KiB | 783.00 KiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/Diagram-to-Infra-code


## Step 3: Generate Dataset

In [3]:
# Generate 500 unique diagrams + 1 copy = 1,000 total images
!python3 data/generate_synthetic.py --output data/synthetic_dataset --count 500 --augmented-copies 1 --seed 42

Generating synthetic dataset:
  Output: data/synthetic_dataset
  Count: 500 unique diagrams
  Augmented copies: 1 per image
  Seed: 42

  Generated 50/500 diagrams (100 total with augmentations)...
  Generated 100/500 diagrams (200 total with augmentations)...
  Generated 150/500 diagrams (300 total with augmentations)...
  Generated 200/500 diagrams (400 total with augmentations)...
  Generated 250/500 diagrams (500 total with augmentations)...
  Generated 300/500 diagrams (600 total with augmentations)...
  Generated 350/500 diagrams (700 total with augmentations)...
  Generated 400/500 diagrams (800 total with augmentations)...
  Generated 450/500 diagrams (900 total with augmentations)...
  Generated 500/500 diagrams (1000 total with augmentations)...

Dataset generation complete!
  Unique diagrams: 500
  Total images (with augmentations): 1000
  Train: data/synthetic_dataset/train/
  Val:   data/synthetic_dataset/val/


## Step 4: Run YOLOv8n Training (Original Run)

Explicit augmentation kwargs:
- `fliplr=0.0` and `flipud=0.0` (disabled horizontal/vertical flips to preserve text & arrow orientation)
- `degrees=10.0`, `translate=0.1`, `scale=0.2`, `mosaic=0.5`, `mixup=0.1`

**Checkpoints are saved to Google Drive:** `/content/drive/MyDrive/diagram-to-infra-runs`

In [ ]:
from ultralytics import YOLO
import sys
sys.path.append('.')
from ml.train import generate_eval_samples

model = YOLO("yolov8n.pt")
results = model.train(
    data="data/dataset.yaml",
    epochs=50,
    batch=16,
    imgsz=640,
    project="/content/drive/MyDrive/diagram-to-infra-runs",
    name="train",
    patience=10,
    degrees=10.0,
    translate=0.1,
    scale=0.2,
    mosaic=0.5,
    mixup=0.1,
    fliplr=0.0,
    flipud=0.0
)

# Capture the dynamic run directory (e.g., train, train2, train3)
run_dir = results.save_dir
print(f"\nTraining completed. Results saved to: {run_dir}")

generate_eval_samples(model, "data/synthetic_dataset/val/images", "ml/eval_samples", num_samples=15)


Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/dataset.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, optim

## Step 4b: Resume Training (Use if Disconnected)
If Colab disconnected, uncomment and run this cell *instead* of Step 4 above to resume from your last checkpoint.

In [ ]:
# from ultralytics import YOLO
# YOLO('/content/drive/MyDrive/diagram-to-infra-runs/train/weights/last.pt').train(resume=True)

## Step 5: Display Training Results & Visual Evaluation Samples

In [ ]:
from IPython.display import Image, display
import glob

print("=== Training Confusion Matrix ===")
display(Image(filename="/content/drive/MyDrive/diagram-to-infra-runs/train/confusion_matrix.png"))

print("=== Results Curves ===")
display(Image(filename="/content/drive/MyDrive/diagram-to-infra-runs/train/results.png"))

print("=== Sample Visual Predictions (from ml/eval_samples/) ===")
eval_images = glob.glob("ml/eval_samples/*.png")[:5]
for img_path in eval_images:
    display(Image(filename=img_path))

## Step 6: (Optional) Push Trained Weights to Hugging Face Hub

In [ ]:
import os
from huggingface_hub import HfApi

# Set your HF_TOKEN here if pushing to HF Hub:
HF_TOKEN = ""  # e.g., "hf_..."
REPO_ID = "parths-29/diagram-to-infra-yolov8n"

if HF_TOKEN:
    api = HfApi()
    api.create_repo(repo_id=REPO_ID, exist_ok=True, token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj="/content/drive/MyDrive/diagram-to-infra-runs/train/weights/best.pt",
        path_in_repo="best.pt",
        repo_id=REPO_ID,
        token=HF_TOKEN
    )
    print(f"Uploaded best.pt to https://huggingface.co/{REPO_ID}")
else:
    print("No HF_TOKEN provided. Weights are saved in Drive at /content/drive/MyDrive/diagram-to-infra-runs/train/weights/best.pt")